In [61]:
import nltk
from nltk.tokenize import PunktSentenceTokenizer, word_tokenize, sent_tokenize
nltk.download('tagsets_json')


[nltk_data] Downloading package tagsets_json to
[nltk_data]     C:\Users\carme\AppData\Roaming\nltk_data...
[nltk_data]   Package tagsets_json is already up-to-date!


True

 # Ejercicio 1


Dada la frase “Juan usa la bicicleta de Clara todos los días soleados.”, se pide realizar
un análisis sintáctico parcial de forma que se obtengan los chunks que se muestran
a continuación. Será necesario, por tanto, definir una gramática con las reglas que
puedan identificar dichos chunks.
Analizando los chunks de las figuras, se puede ver que hay errores. Por ejemplo,
“usa” lo ha etiquetado como adjetivo (JJ) cuando es un verbo. Esto ocurre porque, a
pesar de que se puede especificar el idioma español a la hora de tokenizar con NLTK,
no lo permite cuando se hace el PoS, por eso comete errores en el part-of-speech de
textos en español. Aún así, hay que definir una gramática para que se obtenga la
salida de las figuras. 

In [2]:
sent=  "Juan usa la bicicleta de Clara todos los días soleados."
words_esp= nltk.word_tokenize(sent, language="spanish")
tag_esp=  nltk.pos_tag(words_esp)

traduccion= "Juan uses Clara's bike every sunny day."
words_en= word_tokenize(traduccion)
tag_en= nltk.pos_tag(words_en)
print(tag_en)


[('Juan', 'JJ'), ('uses', 'VBZ'), ('Clara', 'NNP'), ("'s", 'POS'), ('bike', 'NN'), ('every', 'DT'), ('sunny', 'JJ'), ('day', 'NN'), ('.', '.')]


In [25]:
from nltk import RegexpParser, pos_tag

# <NN.*>+ CUBRE DIFERENTES TIPOS DE SUSTANTIVOS
g= r"""
    NP: {<DT>?<JJ>*<NN.*>+} 
    PP: {<PP><NP>}
"""
cp = RegexpParser(g)
tree = cp.parse(tag_esp)

print(tree)

tree_en = cp.parse(tag_en)

print(tree_en)

(S
  (NP Juan/NNP)
  (NP usa/JJ la/NN bicicleta/NN)
  de/IN
  (NP Clara/NNP)
  todos/CC
  (NP los/JJ días/JJ soleados/NN)
  ./.)
(S
  Juan/JJ
  uses/VBZ
  (NP Clara/NNP)
  's/POS
  (NP bike/NN)
  (NP every/DT sunny/JJ day/NN)
  ./.)


# Ejercicio 2

A partir del contenido del fichero de texto “Cycling.txt”, se quiere hacer chunking
para detectar diferentes sintagmas. Utilizar la clase RegexpParser de NLTK para
crear gramáticas que contengan los patrones que se quieren identificar. Se quieren
identificar los siguientes chunks (ponerlo todo en la misma gramática):
- Chunks formados por dos nombres.
- Chunks formados por un determinante y un nombre.
- Chunks formados por una preposición, puede que un determinante y un
nombre.
- Chunks formados por un verbo, un adjetivo y un nombre. 

In [52]:
ruta= "C:/Users/carme/Desktop/PLN/Tema2Hoja3/TextosListadoIII/Cycling.txt"
with open(ruta, "r") as archivo:
    texto= archivo.read() 

token_w= word_tokenize(texto)
tags= pos_tag(token_w)
set_tags= set(t for w, t in tags)

for t in set_tags:
    nltk.help.upenn_tagset(t)
    print("-" * 50)  # Separador

g= r"""
    A: {<NN><NN>}
    B: {<DT><NN>}
    C: {<IN><DT>?<NN>}
    D: {<VB.*><JJ><NN>}
    
"""

cp= RegexpParser(g)
tree= cp.parse(tags)

print(tree)


PRP: pronoun, personal
    hers herself him himself hisself it itself me myself one oneself ours
    ourselves ownself self she thee theirs them themselves they thou thy us
--------------------------------------------------
WDT: WH-determiner
    that what whatever which whichever
--------------------------------------------------
``: opening quotation mark
    ` ``
--------------------------------------------------
.: sentence terminator
    . ! ?
--------------------------------------------------
): closing parenthesis
    ) ] }
--------------------------------------------------
VBG: verb, present participle or gerund
    telegraphing stirring focusing angering judging stalling lactating
    hankerin' alleging veering capping approaching traveling besieging
    encrypting interrupting erasing wincing ...
--------------------------------------------------
NN: noun, common, singular or mass
    common-carrier cabbage knuckle-duster Casino afghan shed thermostat
    investment slide hum

# Ejercicio 3

Utilizando el chunker de NLTK que reconoce Entidades Nombradas (Named Entity,
NE) (https://www.nltk.org/api/nltk.chunk.ne_chunk.html), se pide reconocer las NE
presentes en el texto del archivo “Cycling.txt”.

In [91]:
import nltk
from nltk import word_tokenize, pos_tag, ne_chunk
from nltk.tree import Tree
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')


[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     C:\Users\carme\AppData\Roaming\nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\carme\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\words.zip.


True

In [101]:
token_w=word_tokenize(texto)
tags=pos_tag(token_w)
# Aplicar el chunker para reconocer las entidades nombradas
ne_tree = ne_chunk(tags)
print(ne_tree)

entities=[]

def extraer_entidades(ne_tree):
    for chunk in ne_tree:
        if isinstance(chunk, Tree):
            entity_label= chunk.label()
            entity_words = " ".join([word for word, pos in chunk.leaves()])
            entities.append((entity_words, entity_label))
    return entities

named_entities = extraer_entidades(ne_tree)
for entity, label in named_entities:
    print(f"Entidad: {entity}, Tipo: {label}")
        

(S
  Cycling/VBG
  's/POS
  world/NN
  governing/NN
  body/NN
  ,/,
  the/DT
  (ORGANIZATION UCI/NNP)
  ,/,
  says/VBZ
  it/PRP
  has/VBZ
  no/DT
  plans/NNS
  to/TO
  move/VB
  the/DT
  2025/CD
  Road/NNP
  World/NNP
  Championships/NNP
  away/RB
  from/IN
  (GSP Rwanda/NNP)
  amid/IN
  the/DT
  ongoing/JJ
  conflict/NN
  in/IN
  neighbouring/VBG
  DR/NNP
  Congo/NNP
  ./.
  (PERSON Rwanda/NNP)
  is/VBZ
  set/VBN
  to/TO
  become/VB
  the/DT
  first/JJ
  (GPE African/JJ)
  nation/NN
  to/TO
  host/VB
  the/DT
  event/NN
  from/IN
  21-28/JJ
  September/NNP
  ./.
  The/DT
  (ORGANIZATION M23/NNP)
  rebel/NN
  group/NN
  has/VBZ
  captured/VBN
  almost/RB
  all/DT
  of/IN
  the/DT
  eastern/JJ
  Congolese/NNP
  city/NN
  of/IN
  (GPE Goma/NNP)
  and/CC
  threatened/VBD
  to/TO
  continue/VB
  its/PRP$
  offensive/JJ
  to/TO
  the/DT
  capital/NN
  ,/,
  (GPE Kinshasa/NNP)
  ,/,
  which/WDT
  is/VBZ
  2,600km/CD
  (/(
  1,600/CD
  miles/NNS
  )/)
  away/RB
  ./.
  The/DT
  (ORGANIZATION 

# Ejercicio 4

Mismo ejercicio que el anterior, pero para el contenido del archivo “Health_IA.txt”,
que es texto en español. Probar a hacerlo con la librería NLTK. ¿El resultado es
correcto?

In [123]:
ruta= "C:/Users/carme/Desktop/PLN/Tema2Hoja3/TextosListadoIII/Health_IA.txt"

with open(ruta, "r", encoding='utf-8') as archivo:
    texto= archivo.read() 

token_w= word_tokenize(texto)
tags= pos_tag(token_w)
ne_tree= ne_chunk(tags)
entidades= extraer_entidades(ne_tree)

for entity, label in named_entities:
    print(f"Entidad: {entity}, Tipo: {label}")
        


Entidad: UCI, Tipo: ORGANIZATION
Entidad: Rwanda, Tipo: GSP
Entidad: Rwanda, Tipo: PERSON
Entidad: African, Tipo: GPE
Entidad: M23, Tipo: ORGANIZATION
Entidad: Goma, Tipo: GPE
Entidad: Kinshasa, Tipo: GPE
Entidad: UCI, Tipo: ORGANIZATION
Entidad: Kigali, Tipo: GPE
Entidad: Rwanda, Tipo: GSP
Entidad: UCI, Tipo: ORGANIZATION
Entidad: UCI Road, Tipo: ORGANIZATION
Entidad: Rwanda, Tipo: GSP
Entidad: Switzerland, Tipo: GPE
Entidad: La, Tipo: GPE
Entidad: detecciÃ³n, Tipo: ORGANIZATION
Entidad: ChatGPT, Tipo: ORGANIZATION
Entidad: DeepSeek, Tipo: ORGANIZATION
Entidad: Nuria Ribelles, Tipo: PERSON
Entidad: SecciÃ³n, Tipo: ORGANIZATION
Entidad: Hospital Virgen, Tipo: ORGANIZATION
Entidad: Victoria, Tipo: GPE
Entidad: genÃ³micos, Tipo: ORGANIZATION
Entidad: SecciÃ³n, Tipo: ORGANIZATION
Entidad: EvaluaciÃ³n, Tipo: ORGANIZATION
Entidad: Las, Tipo: PERSON
Entidad: La, Tipo: GPE
Entidad: ChatGPT, Tipo: ORGANIZATION
Entidad: DeepSeek, Tipo: ORGANIZATION
Entidad: Está, Tipo: PERSON
Entidad: Nuria Rib

In [ ]:
No lo hace bien, para español: ¡usar SPACY!


# Ejercicio 5 

Mismo ejercicio que el anterior, pero utilizando spaCy. 

In [125]:
import spacy
nlp = spacy.load("es_core_news_md")


doc= nlp(texto)

for ent in doc.ents: 
    print(f"Entidad: {ent.text}, Tipo: {ent.label_}")


Entidad: ChatGPT, Tipo: MISC
Entidad: DeepSeek, Tipo: MISC
Entidad: Nuria Ribelles, Tipo: PER
Entidad: Sección de Oncología Médica del Hospital Virgen de la Victoria, Tipo: ORG
Entidad: Málaga, Tipo: LOC
Entidad: Sección SEOM de Evaluación de Resultados, Tipo: ORG
